[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AltamarMx/anomalias-2026-2/blob/main/notebooks/021_imputacion_teoria.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
rng = np.random.default_rng(42)
from IPython.display import display, Markdown

# Semana 11A — Imputación de Datos Faltantes: Métodos Deterministas (Teoría)

**Objetivo:** Entender por qué faltan los datos (MCAR/MAR/MNAR),
conocer los métodos deterministas de imputación y sus trade-offs.

**Temas:**

1. Taxonomía de datos faltantes (MCAR, MAR, MNAR)
2. Forward fill / backward fill
3. Interpolación lineal y spline
4. Imputación por día análogo

---
## 1. Taxonomía de datos faltantes

Antes de imputar, hay que entender **por qué** faltan los datos.
El mecanismo determina qué métodos son válidos.

| Tipo | Significado | Ejemplo meteorológico | Consecuencia |
|:---|:---|:---|:---|
| **MCAR** | Missing Completely At Random | Corte eléctrico aleatorio | La distribución observada no está sesgada |
| **MAR** | Missing At Random (depende de variables observadas) | Sensor solar no reporta de noche (depende de `solar_altitude`) | Se puede predecir cuándo falta |
| **MNAR** | Missing Not At Random (depende del valor faltante) | Sensor se satura a T > 40°C y deja de reportar | Los datos observados están **sesgados** (faltan los extremos) |

In [ ]:
# Generar serie completa
n_pts = 500
serie_completa = 25 + 5 * np.sin(2 * np.pi * np.arange(n_pts) / 100) + rng.normal(0, 1.5, n_pts)

# MCAR: eliminar 20% al azar
mask_mcar = rng.random(n_pts) < 0.2
serie_mcar = serie_completa.copy()
serie_mcar[mask_mcar] = np.nan

# MAR: eliminar cuando la señal es baja (simula "noche")
mask_mar = serie_completa < 22
serie_mar = serie_completa.copy()
serie_mar[mask_mar] = np.nan

# MNAR: eliminar cuando el valor es alto (sensor satura)
mask_mnar = serie_completa > 30
serie_mnar = serie_completa.copy()
serie_mnar[mask_mnar] = np.nan

fig1, axes1 = plt.subplots(2, 3, figsize=(14, 7))

configs = [
    (serie_mcar, mask_mcar, "MCAR (20% aleatorio)"),
    (serie_mar, mask_mar, "MAR (falta cuando señal < 22)"),
    (serie_mnar, mask_mnar, "MNAR (falta cuando valor > 30)"),
]

idx = 0
while idx < len(configs):
    serie, mask, titulo = configs[idx]
    # Serie con gaps
    axes1[0, idx].plot(serie_completa, "steelblue", lw=0.5, alpha=0.3, label="Completa")
    axes1[0, idx].plot(serie, "steelblue", lw=0.8)
    axes1[0, idx].scatter(np.where(mask)[0], serie_completa[mask],
                          c="crimson", s=8, alpha=0.5, zorder=5, label="Faltante")
    axes1[0, idx].set_title(titulo)
    axes1[0, idx].legend(fontsize=7)
    axes1[0, idx].set_ylabel("Valor")

    # Histograma: observado vs completo
    obs = serie[~np.isnan(serie)]
    axes1[1, idx].hist(serie_completa, bins=30, density=True, color="steelblue",
                       alpha=0.3, edgecolor="white", label="Completa")
    axes1[1, idx].hist(obs, bins=30, density=True, color="darkorange",
                       alpha=0.6, edgecolor="white", label="Observada")
    axes1[1, idx].set_xlabel("Valor")
    axes1[1, idx].set_ylabel("Densidad")
    axes1[1, idx].legend(fontsize=7)
    idx += 1

plt.suptitle("Efecto del mecanismo de missingness en la distribución observada",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Observa los histogramas:**
> - **MCAR:** La distribución observada (naranja) coincide con la
>   completa (azul). No hay sesgo.
> - **MAR:** Faltan los valores bajos → la distribución observada
>   está sesgada hacia arriba. Pero podemos predecir *cuándo* falta.
> - **MNAR:** Faltan los valores altos → la distribución observada
>   subestima la cola superior. Es el caso más problemático porque
>   no sabemos qué nos estamos perdiendo.

---
## 2. Forward fill / backward fill

El método más simple: reemplazar cada NaN con el último valor
válido (forward) o el siguiente (backward).

- **Ventaja:** Trivial de implementar, rápido.
- **Problema:** Crea escalones artificiales. Para gaps largos,
  el valor "rellenado" puede estar muy lejos del real.

In [ ]:
# Señal sinusoidal + ruido con un gap
n2 = 200
t2 = np.arange(n2)
y2_real = 25 + 5 * np.sin(2 * np.pi * t2 / 50) + rng.normal(0, 0.5, n2)

y2_gap = y2_real.copy()
gap_start, gap_end = 80, 110
y2_gap[gap_start:gap_end] = np.nan

serie2 = pd.Series(y2_gap)
ffill = serie2.ffill()
bfill = serie2.bfill()

fig2, ax2 = plt.subplots(figsize=(12, 4))
ax2.plot(t2, y2_real, "steelblue", lw=1, alpha=0.4, label="Real")
ax2.plot(t2, ffill.values, "crimson", lw=1.5, ls="--", label="Forward fill")
ax2.plot(t2, bfill.values, "darkorange", lw=1.5, ls=":", label="Backward fill")
ax2.axvspan(gap_start, gap_end - 1, color="gray", alpha=0.1)
ax2.set_xlabel("Tiempo")
ax2.set_ylabel("Valor")
ax2.set_title("Forward fill vs backward fill en un gap de 30 pasos")
ax2.legend()
plt.tight_layout()
plt.show()

> El ffill se queda "plano" en el valor pre-gap y no sigue la
> curvatura.  El bfill hace lo mismo pero desde el otro lado.
> Ambos introducen discontinuidades en los bordes del gap.
>
> **Útil para:** gaps de 1–2 registros (minutos).
> **Inadecuado para:** gaps de horas o más.

---
## 3. Interpolación lineal y spline

- **Lineal:** Traza una recta entre los bordes del gap.
  Mejor que ffill pero no captura curvatura.
- **Spline cúbica:** Traza una curva suave ($C^2$ continua).
  Captura curvatura pero puede **oscilar** en gaps largos
  (fenómeno de Runge).

In [ ]:
# Señal sinusoidal con gap corto y gap largo
n3 = 300
t3 = np.arange(n3)
y3_real = 25 + 5 * np.sin(2 * np.pi * t3 / 50) + rng.normal(0, 0.3, n3)

y3_gap = y3_real.copy()
# Gap corto (5 pasos)
y3_gap[50:55] = np.nan
# Gap largo (40 pasos)
y3_gap[150:190] = np.nan

serie3 = pd.Series(y3_gap)
lineal = serie3.interpolate(method="linear")
spline = serie3.interpolate(method="spline", order=3)

fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(13, 4))

# Gap corto
rango_corto = slice(40, 65)
ax3a.plot(t3[rango_corto], y3_real[rango_corto], "steelblue", lw=2, label="Real")
ax3a.plot(t3[rango_corto], lineal.values[rango_corto], "crimson", lw=1.5, ls="--", label="Lineal")
ax3a.plot(t3[rango_corto], spline.values[rango_corto], "seagreen", lw=1.5, ls=":", label="Spline")
ax3a.axvspan(50, 54, color="gray", alpha=0.1)
ax3a.set_title("Gap corto (5 pasos)")
ax3a.set_xlabel("Tiempo")
ax3a.set_ylabel("Valor")
ax3a.legend(fontsize=9)

# Gap largo
rango_largo = slice(135, 205)
ax3b.plot(t3[rango_largo], y3_real[rango_largo], "steelblue", lw=2, label="Real")
ax3b.plot(t3[rango_largo], lineal.values[rango_largo], "crimson", lw=1.5, ls="--", label="Lineal")
ax3b.plot(t3[rango_largo], spline.values[rango_largo], "seagreen", lw=1.5, ls=":", label="Spline")
ax3b.axvspan(150, 189, color="gray", alpha=0.1)
ax3b.set_title("Gap largo (40 pasos)")
ax3b.set_xlabel("Tiempo")
ax3b.legend(fontsize=9)

plt.suptitle("Interpolación lineal vs spline cúbica", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Gap corto:** Ambos métodos funcionan bien — la diferencia es mínima.
>
> **Gap largo:** La interpolación lineal traza una recta que no sigue
> la curvatura.  La spline sigue mejor la forma pero puede **oscilar**
> en los extremos del gap, produciendo valores irreales.
>
> Ninguno captura el **ciclo diario** si el gap es de horas/días.

---
## 4. Imputación por día análogo

Para datos meteorológicos con **estacionalidad diaria**:

> Para cada timestamp faltante, buscar el mismo horario en días
> cercanos (±5–7 días) del mismo mes. Promediar los valores encontrados.

**Ventajas:**
- Explota la estacionalidad diaria
- Funciona para gaps de horas a días
- No asume modelo paramétrico

**Limitaciones:**
- Necesita suficientes días análogos disponibles
- Introduce el ruido de los días de referencia
- No funciona para gaps de semanas

In [ ]:
# Simular 30 días de datos horarios con ciclo diario
n_horas = 24 * 30
t_horas = np.arange(n_horas)
ciclo_diario = 5 * np.sin(2 * np.pi * (t_horas % 24 - 6) / 24)
y4_real = 22 + ciclo_diario + rng.normal(0, 0.8, n_horas)

y4_gap = y4_real.copy()
# Gap de 2 días (horas 240 a 288 = día 10 y 11)
gap_s4, gap_e4 = 240, 288
y4_gap[gap_s4:gap_e4] = np.nan

serie4 = pd.Series(y4_gap, index=pd.date_range("2024-07-01", periods=n_horas, freq="h"))

# Día análogo: promediar misma hora en ±5 días
y4_analogo = y4_gap.copy()
hora_idx = 0
while hora_idx < (gap_e4 - gap_s4):
    pos = gap_s4 + hora_idx
    hora_del_dia = pos % 24
    # Buscar misma hora en días ±1..5 antes y después del gap
    valores_ref = []
    dia_offset = 1
    while dia_offset <= 5:
        idx_antes = pos - dia_offset * 24
        idx_despues = pos + dia_offset * 24
        if 0 <= idx_antes < len(y4_gap) and not np.isnan(y4_gap[idx_antes]):
            valores_ref.append(y4_gap[idx_antes])
        if 0 <= idx_despues < len(y4_gap) and not np.isnan(y4_gap[idx_despues]):
            valores_ref.append(y4_gap[idx_despues])
        dia_offset += 1
    if len(valores_ref) > 0:
        y4_analogo[pos] = np.mean(valores_ref)
    hora_idx += 1

# Comparar con interpolación lineal
serie4_lineal = serie4.interpolate(method="linear")

fig4, ax4 = plt.subplots(figsize=(13, 5))
rango4 = slice(gap_s4 - 24, gap_e4 + 24)
ax4.plot(t_horas[rango4], y4_real[rango4], "steelblue", lw=2, label="Real")
ax4.plot(t_horas[rango4], serie4_lineal.values[rango4], "crimson",
         lw=1.5, ls="--", label="Interpolación lineal")
ax4.plot(t_horas[rango4], y4_analogo[rango4], "seagreen",
         lw=1.5, ls="-", label="Día análogo (±5 días)")
ax4.axvspan(gap_s4, gap_e4 - 1, color="gray", alpha=0.1, label="Gap (2 días)")
ax4.set_xlabel("Hora")
ax4.set_ylabel("Temperatura (°C)")
ax4.set_title("Día análogo vs interpolación lineal — gap de 2 días")
ax4.legend()
plt.tight_layout()
plt.show()

> **Resultado:**
> - La **interpolación lineal** traza una recta que ignora el ciclo diario.
> - El **día análogo** reproduce el patrón diario (pico a las ~14h,
>   valle a las ~6h) porque copia de días reales cercanos.
>
> Para gaps largos con estacionalidad, el día análogo es claramente
> superior a la interpolación.  Pero para gaps muy largos (semanas),
> necesitaremos modelos que capturen la dinámica temporal
> → ARIMA/SARIMA (semanas 12–13).

---
## Resumen

| Método | Idea | Mejor para | Problema principal |
|:---|:---|:---|:---|
| **Forward/backward fill** | Copiar último/siguiente valor | Gaps de 1–2 registros | Escalones artificiales |
| **Interpolación lineal** | Recta entre bordes del gap | Gaps cortos sin curvatura | No sigue ciclos |
| **Spline cúbica** | Curva suave entre bordes | Gaps cortos/medianos | Puede oscilar en gaps largos |
| **Día análogo** | Promediar misma hora en días cercanos | Gaps de horas a días con estacionalidad | Introduce ruido, necesita días disponibles |

**Próxima sesión (11B):** Evaluar cuantitativamente estos métodos
sobre `tdb` de ClimaLab con gaps artificiales y métricas RMSE/MAE.